# Lab 3: The Tidyverse and Data Visualization

<img src="ds4all-logo.png" alt="Data Science for All Logo" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

Welcome to Lab 3 of *Data Science for All*!

Last week you wrote your own functions and made your code branch and repeat, all in
base R. This week we pick up the **tidyverse**, the set of packages we will use for
the rest of the semester, and then spend most of our time making plots.

This lab will be due on **Thursday at 5 PM (Week 4)**. As always, collaborating on
labs is encouraged, but don't just share answers.

## Today's lab

In today's lab, you'll learn how to:

1. load the tidyverse and read a CSV that lives in a folder inside your project;
2. tell an **absolute path** from a **relative path**, and know why relative paths are the ones that survive being shared;
3. build scatter plots, histograms, bar charts, and line charts with `ggplot2`;
4. start from a research question and pick the plot that answers it; and
5. work out the relative path an image needs so that it shows up in a blog post.

Number 4 is the one that matters for your blog. Before you test anything, you should
be able to look at a picture of your data and say whether it has a shot at answering
your question.

In [ ]:
# Initialize the autograder and JupyterHub helper
source("helpers.R")

library(tidyverse)
library(palmerpenguins)

options(repr.matrix.max.rows = 20)
options(repr.plot.width = 7, repr.plot.height = 4.5)

After you run this, you'll see a big red message here. It looks quite alarming. It's not an error message but it's information about the packages loaded. Sometimes you will have two packages with functions that are the same name. R will try to resolve which one gets called over the other. If one is called first we say that function 'masks' another function. Moreover, any variable can be masked. Here a dataset is masked: the package `palmerpenguins` has a variables `penguins` which masks the default base R `penguins`. 

<br/><br/>

<hr style="border: 1px solid #fdb515;" />

## 1. Importing

Two things get imported in a typical analysis: **packages** and **data**. 

### Packages

A package is a bundle of functions somebody else wrote. You can always write functions and put them into a script `helpers.R` and create something like a package.

In this notebook, `source("helpers.R")` already put somebody else's functions into your session. So what makes a **package** different?

- **You load it by name, not by path.** `library(readr)` works wherever the package happens to be installed. `source()` needs the file to be exactly where you said it is.
- **It has a version, and a list of what it depends on.** That is how `library(tidyverse)` pulls in eight other packages on its own, and how code you write today has a chance of still running next year.
- **Every function has a help page.** Try `?read_csv`. There is no help page for a function you sourced.
- **It controls its own names.** A package declares which functions it hands out, and `readr::read_csv()` says which one you mean when two packages define the same name. You have already seen that syntax in `ottr::check()`.

A bundle of functions is the starting point. A package is that bundle plus a name, a version, documentation, declared dependencies, and control over its own namespace, wrapped up so that anyone can install it and use it without knowing where the files ended up.

The three we use today:

| Package | What it gives us |
|---|---|
| `readr` | `read_csv()`, for getting data off the disk and into R |
| `dplyr` | `rename()`, `count()`, `filter()`, for reshaping a table |
| `ggplot2` | `ggplot()` and the `geom_*()` functions, for plotting |

You load a package once per session, and you have to do it every session. `library(tidyverse)` will be a typical thing you see at the top of these notebooks. 

### Where are we?

R is always sitting in some folder. That folder is the **working directory**, and
every file you ask for gets looked up starting from there. Two functions tell you
what R can see:

In [ ]:
getwd()        # the folder R is sitting in right now

In [ ]:
list.files()   # everything in that folder

You should see a folder called `data` in that list. Look inside it:

In [ ]:
list.files("data")

### Absolute and relative paths

A **path** is the address of a file. There are two ways to write one.

An **absolute path** starts at the very top of the file system and spells out every
step. On JupyterHub it looks like

```
/home/jovyan/lab03/data/baby.csv
```

and on a Windows laptop like

```
C:/Users/sarah/Documents/dats1001/lab03/data/baby.csv
```

A **relative path** starts from wherever R is right now, and only says how to get to
the file from there:

```
data/baby.csv
```

The pieces that show up in relative paths:

| Piece | Meaning |
|---|---|
| `data/baby.csv` | the file `baby.csv`, inside the folder `data`, inside the current folder |
| `./baby.csv` | `baby.csv` in the current folder. The `./` is optional |
| `../baby.csv` | go **up** one folder, then look for `baby.csv` |
| `/home/jovyan/...` | absolute. That leading `/` means "start at the very top" |

**Use relative paths.** An absolute path contains your username and your folder
layout, so it works on exactly one computer: yours. The moment you hand the file to a
classmate, submit it to Gradescope, or push it to your blog repository, an absolute
path breaks. A relative path keeps working as long as the file is still in the same
place relative to your notebook.

<div style="border-left: 4px solid #fdb515; background: #fffdf5; padding: 0.8em 1em; margin: 1em 0;">
<b>This rule comes back in your blog.</b>
<p>When you put an image in a blog post, the same logic applies. The computer has no
idea where the image is unless you tell it, and it will look for exactly the address
you wrote, starting from the file doing the pointing. Write
<code>/Users/you/Desktop/photo.jpg</code> and it will work on your laptop and be a
broken box for every reader.</p>
<p>The syntax for an image in a <code>.qmd</code> file is
<code>![alt text](path)</code>, and <code>path</code> should be defined relatively.</p>
</div>

### The structure I expect from your blog

```
myblog/                 # your project folder
├─ _quarto.yml
├─ index.qmd
├─ about.qmd
├─ posts/               # folder where posts live
│  ├─ _metadata.yml
│  ├─ post1/            # new sub-directory for the post
│  │  └─ index.qmd      # the actual post file
│  └─ ...               # (other posts go here eventually)
└─ is_me.jpeg           # the image for your About page
```

Notice that nothing here has an absolute path in it. Every file finds every other file
by their positions in this tree, which is why the whole thing still works after you
push it to GitHub and it gets rebuilt on a machine that is not yours.

Two positions worth noticing, because we come back to both:

- `about.qmd` and `is_me.jpeg` are both at the top level, sitting next to each other.
- `posts/post1/index.qmd` is **two** folders down from the top.

### Reading data

`read_csv()` takes a path and gives back a **tibble**, the tidyverse's version of a
data table. We pass `show_col_types = FALSE` to keep it from printing a summary of
every column each time.

---

**Question 1.1.** Read `data/baby.csv` into a tibble named `baby`. Use a relative path,
and pass `show_col_types = FALSE`.

This is a real dataset: 1,174 mother and newborn pairs from the Child Health and
Development Studies, which followed pregnancies among Kaiser Foundation Health Plan
members in Oakland, California. We come back to it later in the semester.

In [ ]:
baby <- NULL # YOUR CODE HERE

baby

In [ ]:
. <- ottr::check("tests/q1_1.R")

### Column names with spaces

Look at those column names. `Birth Weight` has a space in it. R will not let you write
`baby$Birth Weight`, because it reads the space as the end of the name.

There are two ways to deal with this.

**Backticks.** Wrap any awkward name in backticks and R treats the whole thing as one
name: `` `Birth Weight` ``. You need this for names with spaces, names with
punctuation, and names that start with a number.

**Rename the columns once, at the top.** Better. `rename()` takes
`new_name = old_name` pairs and hands back a tibble with the names fixed. Do it right
after you read the file, and nothing downstream has to think about it again.

Here is the worked example on `baby`. The new names are all lowercase with
underscores, which is the tidyverse convention and much easier to type.

In [ ]:
baby <- baby |>
  rename(
    birth_weight              = `Birth Weight`,
    gestational_days          = `Gestational Days`,
    maternal_age              = `Maternal Age`,
    maternal_height           = `Maternal Height`,
    maternal_pregnancy_weight = `Maternal Pregnancy Weight`,
    maternal_smoker           = `Maternal Smoker`
  )

baby

<div style="border-left: 4px solid #fdb515; background: #fffdf5; padding: 0.8em 1em; margin: 1em 0;">
<b>Run that cell once.</b>
<p>It renames the columns, so running it a second time will throw
<code>Column `Birth Weight` doesn't exist</code>. That error is harmless, and it means
the rename already worked. If you want a clean slate, restart the kernel and run the
notebook from the top to reload the original.</p>
</div>

The columns, with units:

| Column | Meaning |
|---|---|
| `birth_weight` | baby's weight at birth, in **ounces** |
| `gestational_days` | length of the pregnancy, in days |
| `maternal_age` | mother's age, in years |
| `maternal_height` | mother's height, in inches |
| `maternal_pregnancy_weight` | mother's weight before pregnancy, in pounds |
| `maternal_smoker` | `TRUE` if the mother smoked during pregnancy |

---

**Question 1.2.** Now you do one.

Read `data/movies/top_movies_2017.csv` into a tibble named `top_movies`, then rename its
columns. This is the 200 highest grossing films in the US, one row per film.

The original names are `Title`, `Studio`, `Gross`, `Gross (Adjusted)`, and `Year`.
Rename them to `title`, `studio`, `gross`, `gross_adjusted`, and `year`.

*Hint:* `Gross (Adjusted)` has a space **and** parentheses, so it needs backticks.

In [ ]:
top_movies <- read_csv("...", show_col_types = FALSE) |>
  rename(
    title          = Title,
    studio         = ...,
    gross          = ...,
    gross_adjusted = ...,
    year           = ...
  )

top_movies

In [ ]:
. <- ottr::check("tests/q1_2.R")

We need one more table. This one is already summarized: one row per year, with the
combined box office for that year. Run the cell, no changes needed.

In [ ]:
movies_by_year <- read_csv("data/movies/movies_by_year.csv", show_col_types = FALSE) |>
  rename(
    year        = Year,
    total_gross = `Total Gross`,
    n_movies    = `Number of Movies`,
    top_movie   = `#1 Movie`
  )

movies_by_year

---

**Question 1.3.** Look back at the blog tree above. You are writing `about.qmd`, and
you want to show `is_me.jpeg` on that page.

Which path goes inside `about.qmd`?

- `A.` `/Users/me/Documents/myblog/is_me.jpeg`, the full address starting from the top of the file system
- `B.` `myblog/is_me.jpeg`, naming the project folder first and then the file inside it
- `C.` `is_me.jpeg`, since `about.qmd` and the image are sitting in the same folder
- `D.` `../is_me.jpeg`, going up one level out of the folder that `about.qmd` is in

Set `path_answer` to the letter of the correct choice, as a string.

*Hint:* the path is read starting from the folder the file doing the pointing is in.
Where is `about.qmd`?

In [ ]:
path_answer <- NULL # YOUR CODE HERE

path_answer

In [ ]:
. <- ottr::check("tests/q1_3.R")

### Look at your data before you plot it

You cannot plot a column whose name you guessed wrong, and R is fussy about spelling
and capitalization. Check first. Three functions do this:

In [ ]:
baby |> head(5)     # first five rows

In [ ]:
names(baby)         # just the column names

In [ ]:
glimpse(baby)       # names, types, and the first few values of each

`glimpse()` is the most useful of the three. The tags next to each column,
`<dbl>` for numbers and `<lgl>` for logical, tell you how R is treating that
column, and that determines which plots are even possible.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />

## 2. Warm-up: the grammar of graphics

Every `ggplot2` plot is built from the same three pieces, added together with `+`:

1. **Data.** The tibble you are plotting. `ggplot(penguins)`
2. **Aesthetics.** Which column goes on which axis, or controls color, or size.
   `aes(x = flipper_length_mm, y = body_mass_g)`
3. **Geom.** The shape used to draw each observation. `geom_point()`

Data and aesthetics alone get you an empty set of axes. R knows the range of the
data, but you have not told it what to draw:

In [ ]:
penguins <- penguins |>
  drop_na(flipper_length_mm, body_mass_g)

ggplot(penguins) +
  aes(x = flipper_length_mm, y = body_mass_g)

Add a geom and the points appear:

In [ ]:
ggplot(penguins) +
  aes(x = flipper_length_mm, y = body_mass_g) +
  geom_point(alpha = 0.5)

Use `names()` to see the column names in the dataset:

In [ ]:
names(penguins)

<div style="border-left: 4px solid #fdb515; background: #fffdf5; padding: 0.8em 1em; margin: 1em 0;">
<b>Labeling standards</b>

Every plot you turn in, in this lab and in your blog, gets all four of these. The autograder checks the title and the axis labels. I read the rest.

- **A title that names the variables or states the finding, and never names the chart type.** "Birth weight vs length of pregnancy" is fine. "Longer pregnancies produce heavier babies" is better, because it tells the reader what to look for. "Scatter plot of gestational days and birth weight" is not a title: the reader can already see it is a scatter plot, and `gestational_days` is a column name, not words.
- **Axis labels in plain words, with units.** `birth_weight` is a column name. "Birth weight (ounces)" is a label. If the variable has units, put them in parentheses. If it genuinely has none, leave them off rather than inventing some.
- **A legend title, whenever there is a legend.** Set it using the name of the aesthetic that made it: `color = "Species"`, `fill = "Drivetrain"`. An untitled legend makes the reader guess what the colors mean.
- **A caption naming the source.** `caption = "Source: Child Health and Development Studies"`. Where the numbers came from is part of the plot, not an optional extra.

Here is the test for all four. Cover the code and show someone nothing but the picture. If they cannot tell you what they are looking at and where the data came from, the labels are not finished.
</div>

---

**Question 2.1.** Make the same plot, but color the points by `species`, and finish it
with `labs()` and `theme_minimal()`.

Fill in the blanks. Notice that `color = species` goes **inside** `aes()`, because it
is mapped to a column. A fixed color like `color = "chartreuse4"` would go inside
`geom_point()` instead, because it is not mapped to anything.

In [ ]:
penguin_scatter <- ggplot(penguins) +
  aes(x = flipper_length_mm, y = ..., color = ...) +
  geom_point(alpha = 0.5) +
  labs(
    title = "...",
    x = "Flipper length (mm)",
    y = "...",
    color = "Species",
    caption = "Source: palmerpenguins"
  ) +
  theme_minimal(base_size = 16)

penguin_scatter

In [ ]:
. <- ottr::check("tests/q2_1.R")

A plot is an R object like any other variable we create. `penguin_scatter` is saved, so you can add to it
later without retyping the whole thing:

In [ ]:
penguin_scatter + facet_wrap(~ species)

<br/><br/>

<hr style="border: 1px solid #fdb515;" />

## 3. Scatter plots: is there an association between two variables?

**Use a scatter plot when both variables are numeric and you want to know whether
they move together.**

This is the plot that answers *"is there an association between X and Y?"*, which is
one of the two questions most of you will ask of your blog data. Plot it before you
test it. If the cloud of points has no shape at all, no test is going to find one.

Worked example. Does a longer pregnancy produce a heavier baby?

In [ ]:
ggplot(baby) +
  aes(x = gestational_days, y = birth_weight) +
  geom_point(alpha = 0.5, color = "chartreuse4") +
  labs(
    title = "Longer pregnancies produce heavier babies",
    x = "Length of pregnancy (days)",
    y = "Birth weight (ounces)",
    caption = "Source: Child Health and Development Studies"
  ) +
  theme_minimal(base_size = 16)

The points drift up from left to right, so longer pregnancies tend to go with
heavier babies. That is a **positive association**. Three shapes to look for:

- **Positive.** Points rise left to right. High X goes with high Y.
- **Negative.** Points fall left to right. High X goes with low Y.
- **None.** A shapeless blob. Knowing X tells you nothing useful about Y.

Notice how much scatter there is even here. The trend is real, but a given number of
days is nowhere near enough to predict a particular baby's weight. "There is an
association" and "X determines Y" are very different claims.

Association is also not causation. The plot tells you that two columns move together
and nothing at all about why.

---

**Question 3.1.** Add a third variable. Color the points by `maternal_smoker` to see
whether the smoker and nonsmoker clouds sit in different places.

Fill in the blanks.

In [ ]:
smoker_scatter <- ggplot(baby) +
  aes(x = ..., y = ..., color = ...) +
  geom_point(alpha = 0.5) +
  labs(
    title = "Length of pregnancy and birth weight, by smoking status",
    x = "Length of pregnancy (days)",
    y = "Birth weight (ounces)",
    color = "Smoked during pregnancy",
    caption = "Source: Child Health and Development Studies"
  ) +
  theme_minimal(base_size = 16)

smoker_scatter

In [ ]:
. <- ottr::check("tests/q3_1.R")

---

**Question 3.2.** Now build one from scratch.

Make a scatter plot with the mother's age (`maternal_age`) on the x-axis and the
baby's birth weight (`birth_weight`) on the y-axis. Name it `age_scatter`. Give it a
title, both axis labels, and `theme_minimal(base_size = 16)`.

Use `alpha = 0.5` on the points. There are 1,174 babies and many land on exactly the
same spot, so without transparency you cannot tell how many points are stacked up.

In [ ]:
age_scatter <- NULL # YOUR CODE HERE

age_scatter

In [ ]:
. <- ottr::check("tests/q3_2.R")

---

**Question 3.3.** Compare `age_scatter` to the pregnancy-length plot above. Which
statement describes `age_scatter` best?

- `A.` Strong positive association: older mothers clearly have heavier babies.
- `B.` Strong negative association: older mothers clearly have lighter babies.
- `C.` The plot is unreadable, so nothing can be said about these two variables.
- `D.` Essentially no association: the cloud is a shapeless blob, so a mother's age tells you almost nothing about her baby's birth weight, and this pair would not be a promising place to look for a relationship.

Set `scatter_answer` to the letter, as a string.

This is the useful outcome, not a failure. Finding out now that two variables are
unrelated saves you from building a blog post on top of a relationship that is not
there.

In [ ]:
scatter_answer <- NULL # YOUR CODE HERE

scatter_answer

In [ ]:
. <- ottr::check("tests/q3_3.R")

<br/><br/>

<hr style="border: 1px solid #fdb515;" />

## 4. Histograms: what does one variable look like, and do two groups differ?

**Use a histogram when you have _one numeric variable_ and you want to see its
distribution.**

A histogram chops the range of the variable into bins and draws a bar for each, where
the height is how many observations landed in that bin. It answers: where is the bulk
of the data, is it symmetric or lopsided, is there more than one clump, are there
outliers.

Only `x` goes in `aes()`. `ggplot2` counts the rows for the y-axis itself.

In [ ]:
ggplot(baby) +
  aes(x = birth_weight) +
  geom_histogram(bins = 30, fill = "chartreuse4", color = "white") +
  labs(
    title = "Birth weights",
    x = "Birth weight (ounces)",
    y = "Count",
    caption = "Source: Child Health and Development Studies"
  ) +
  theme_minimal(base_size = 16)

`fill` is the color inside the bars, `color` is the outline. That distinction trips
people up constantly:

- **`color`** is for things drawn with a line: points, lines, text, and outlines.
- **`fill`** is for things with an interior: bars, boxes, and areas.

`bins` controls how many bars you get. The default is 30, which is rarely the right
number for your data. Too few and you flatten real structure, too many and you are
looking at noise. Change it and see:

In [ ]:
ggplot(baby) +
  aes(x = birth_weight) +
  geom_histogram(bins = 5, fill = "chartreuse4", color = "white") +
  labs(title = "5 bins: too coarse", x = "Birth weight (ounces)", y = "Count") +
  theme_minimal(base_size = 16)

In [ ]:
ggplot(baby) +
  aes(x = birth_weight) +
  geom_histogram(bins = 120, fill = "chartreuse4", color = "white") +
  labs(title = "120 bins: too fine", x = "Birth weight (ounces)", y = "Count") +
  theme_minimal(base_size = 16)

### Comparing two groups

Here is the other question a histogram answers, which will be helpful to think about for the blog: **do two groups differ?**

Put the numeric variable on `x` as usual, then split by the grouping variable. Two
ways to do it.

**Facet.** One panel per group, stacked so the x-axes line up.

In [ ]:
ggplot(baby) +
  aes(x = gestational_days) +
  geom_histogram(bins = 30, fill = "chartreuse4", color = "white") +
  facet_wrap(~ maternal_smoker, ncol = 1) +
  labs(
    title = "Length of pregnancy, by smoking status",
    x = "Length of pregnancy (days)",
    y = "Count",
    caption = "Source: Child Health and Development Studies"
  ) +
  theme_minimal(base_size = 16)

**Overlay.** Both groups in one panel, told apart by `fill`. You need
`position = "identity"` so the bars sit on top of each other instead of stacking, and
`alpha` so you can see through them.

In [ ]:
ggplot(baby) +
  aes(x = gestational_days, fill = maternal_smoker) +
  geom_histogram(bins = 30, position = "identity", alpha = 0.5, color = "white") +
  labs(
    title = "Length of pregnancy, by smoking status",
    x = "Length of pregnancy (days)",
    y = "Count",
    fill = "Smoked during pregnancy",
    caption = "Source: Child Health and Development Studies"
  ) +
  theme_minimal(base_size = 16)

Faceting is easier to read with three or more groups. Overlaying is better for two,
because the overlap itself is the point: if the two humps sit almost on top of each
other, the groups are not very different.

One catch with facets. The two groups here are different sizes, 715 nonsmokers and
459 smokers, so the taller panel is partly just the bigger group. When group sizes
differ a lot and you care about **shape** rather than counts, plot density instead of
count by adding `y = after_stat(density)` to the `aes()`.

So far, we're using the logical TRUE / FALSE values in the variable `maternal_smoker`. In the tidyverse, there is another data type called a `factor` which can be used to create categories with multiple items (not just two). We could redefine the `maternal_smoker` variable

In [ ]:
baby_fct <- baby |>
  mutate(maternal_smoker = factor(maternal_smoker,
                                  levels = c(FALSE, TRUE),
                                  labels = c("Nonsmoker", "Smoker")))

glimpse(baby_fct)

Now the groups have labels with better semantics:

In [ ]:
head(baby_fct)

In [ ]:
ggplot(baby_fct) +
  aes(x = gestational_days) +
  geom_histogram(bins = 30, fill = "chartreuse4", color = "white") +
  facet_wrap(~ maternal_smoker, ncol = 1) +
  labs(
    title = "Length of pregnancy, by smoking status",
    x = "Length of pregnancy (days)",
    y = "Count",
    caption = "Source: Child Health and Development Studies"
  ) +
  theme_minimal(base_size = 16)

---

**Question 4.1.** Make a histogram of the mothers' heights (`maternal_height`), with
25 bins, filled in a color of your choice. Name it `height_hist`.

Fill in the blanks.

In [ ]:
height_hist <- ggplot(baby) +
  aes(x = ...) +
  geom_histogram(bins = ..., fill = "...", color = "white") +
  labs(
    title = "...",
    x = "Mother's height (inches)",
    y = "..."
  ) +
  theme_minimal(base_size = 16)

height_hist

In [ ]:
. <- ottr::check("tests/q4_1.R")

---

**Question 4.2.** Now from scratch, make the plot to assess

*Do babies of mothers who smoked during pregnancy weigh less than babies of mothers
who did not?*

Build a histogram of `birth_weight` split by `maternal_smoker`, and name it
`smoker_hist`. Use either faceting or overlaying, whichever you think reads better.
Give it a title, axis labels, and a theme.

In [ ]:
smoker_hist <- NULL # YOUR CODE HERE

smoker_hist

In [ ]:
. <- ottr::check("tests/q4_2.R")

---

**Question 4.3.** Based on `smoker_hist`, which statement is best supported?

- `A.` The two distributions are identical, so smoking has no relationship to birth weight.
- `B.` Every baby of a smoker weighs less than every baby of a nonsmoker.
- `C.` Babies of smokers weigh more on average than babies of nonsmokers.
- `D.` The smokers' distribution sits noticeably to the left of the nonsmokers', so babies of smokers tend to weigh less, but the two distributions overlap a great deal and plenty of individual smokers' babies are heavier than plenty of nonsmokers'.

Set `group_answer` to the letter, as a string.

That gap is about nine ounces on average. Whether a gap that size, in a sample this
size, is more than chance is exactly the question we take up later in the semester.
The plot is what tells you the question is worth asking.

In [ ]:
group_answer <- NULL # YOUR CODE HERE

group_answer

In [ ]:
. <- ottr::check("tests/q4_3.R")

<br/><br/>

<hr style="border: 1px solid #fdb515;" />

## 5. Bar charts: comparing values across categories

**Use a bar chart when your x-axis is categorical.**

A histogram and a bar chart look similar and are not the same thing. A histogram bins
a numeric variable, so the bars touch and the left-to-right order is fixed by the
numbers. A bar chart has one bar per category, with gaps between them, and the order
is whatever you choose.

We switch datasets here, to `top_movies`: the 200 highest grossing US films, one row
per film.

There are two geoms, and picking the wrong one is the most common bar chart error.

`geom_bar()` counts the rows for you. Give it raw data and only an `x`:

In [ ]:
ggplot(top_movies) +
  aes(x = studio) +
  geom_bar(fill = "chartreuse4") +
  labs(
    title = "Films in the top 200, by studio",
    x = "Studio",
    y = "Number of films",
    caption = "Source: Data 8, top_movies_2017"
  ) +
  theme_minimal(base_size = 16)

Unreadable. Twenty-three studios do not fit across a page, and the labels collide.
Two fixes, and you want both.

`coord_flip()` turns the chart on its side so labels have room, and
`fct_infreq()` orders the categories by how often they occur instead of
alphabetically. Alphabetical is almost never the order a reader wants.

In [ ]:
ggplot(top_movies) +
  aes(x = fct_rev(fct_infreq(studio))) +
  geom_bar(fill = "chartreuse4") +
  coord_flip() +
  labs(
    title = "Films in the top 200, by studio",
    x = "Studio",
    y = "Number of films",
    caption = "Source: Data 8, top_movies_2017"
  ) +
  theme_minimal(base_size = 16)

Here we've used `geom_bar()` which does the counting for us. The more general `geom_col()` does no counting at all. Use it when you have **already** summarized the
data and have a column holding the height of each bar. `count()` is the tidyverse
function that does that summarizing:

In [ ]:
studio_counts <- top_movies |>
  count(studio, name = "n")

studio_counts

In [ ]:
ggplot(studio_counts) +
  aes(x = fct_reorder(studio, n), y = n) +
  geom_col(fill = "chartreuse4") +
  coord_flip() +
  labs(
    title = "Films in the top 200, by studio",
    x = "Studio",
    y = "Number of films",
    caption = "Source: Data 8, top_movies_2017"
  ) +
  theme_minimal(base_size = 16)

Same picture, two routes. The rule:

| You have | Use |
|---|---|
| Raw data, one row per observation | `geom_bar()`, with only `x` |
| Summarized data, one row per category plus a value column | `geom_col()`, with `x` **and** `y` |

If you ever get "`stat_count()` can only have an `x` or `y` aesthetic", you gave
`geom_bar()` a `y`. You wanted `geom_col()`.

---

**Question 5.1.** The `baby` table has a categorical column, `maternal_smoker`. Make a
bar chart showing how many mothers are in each group, using the **raw** `baby` data
and the geom that counts for you. Name it `smoker_bar`.

Fill in the blanks.

In [ ]:
smoker_bar <- ggplot(baby) +
  aes(x = ...) +
  geom_...(fill = "chartreuse4") +
  labs(
    title = "...",
    x = "Smoked during pregnancy",
    y = "..."
  ) +
  theme_minimal(base_size = 16)

smoker_bar

In [ ]:
. <- ottr::check("tests/q5_1.R")

---

**Question 5.2.** Now the summarized route, from scratch.

The `movies_by_year` table already has one row per year and a `total_gross` column
holding that year's combined box office, in millions of dollars. Nothing needs
counting.

Make a bar chart of `total_gross` by `year`, using the geom that does **not** count,
and name it `gross_bar`. (So `total_gross` is replacing the usual `count`). Give it a title, axis labels, and a theme.

In [ ]:
gross_bar <- NULL # YOUR CODE HERE

gross_bar

In [ ]:
. <- ottr::check("tests/q5_2.R")

<br/><br/>

<hr style="border: 1px solid #fdb515;" />

## 6. Line charts: trends over time

**Use a line chart when the x-axis is a date or a time and you want to see a trend.**

The line says that consecutive points are connected, which only means something when
there is a real order to them. Never connect categories with a line.

`movies_by_year` has one row per year from 1980 to 2015. Check the types first:

In [ ]:
glimpse(movies_by_year)

`year` is `<dbl>`, a plain number. That is fine here: years are ordered, evenly
spaced, and R will lay them out correctly on the axis.

You will also meet real date columns, tagged `<date>`. `read_csv()` recognizes dates
written as `YYYY-MM-DD` and converts them automatically. If a date column ever comes
in as `<chr>`, your x-axis will be a jumble of evenly spaced text labels, and you need
to convert it with something like `as.Date()` before plotting.

Here is the same `gross_bar` data as a line:

In [ ]:
ggplot(movies_by_year) +
  aes(x = year, y = total_gross) +
  geom_line(linewidth = 1) +
  labs(
    title = "US box office has grown steadily since 1980",
    x = "Year",
    y = "Total gross (millions of dollars)",
    caption = "Source: Data 8, movies_by_year"
  ) +
  theme_minimal(base_size = 16)

`linewidth = 1` is worth adding every time. The default line is thin enough to
disappear on a projector.

Compare this to `gross_bar` from Question 5.2. Both are correct. The line makes the
trend easier to follow, because your eye tracks a continuous shape faster than it
compares 36 bar heights. When time is on the x-axis, use the line plot instead. 

### A warning about two series on one plot

It is tempting to put two lines on the same axes. That works only when the two share
units and sit on a similar scale. Here is what happens when they do not. `n_movies`
is a count of a few hundred, `total_gross` is thousands of millions:

In [ ]:
ggplot(movies_by_year) +
  aes(x = year) +
  geom_line(aes(y = total_gross), linewidth = 1) +
  geom_line(aes(y = n_movies), linewidth = 1) +
  labs(title = "Do not do this", x = "Year", y = "???") +
  theme_minimal(base_size = 16)

The number of movies is squashed flat against the bottom, and the y-axis label
has no honest value, because the two lines do not share a unit. When this
happens, facet the two series, or plot them separately, or standardize them first.

---

**Question 6.1.** Make a line chart of the **number of movies released** (`n_movies`)
over time, and name it `movies_line`.

Fill in the blanks.

In [ ]:
movies_line <- ggplot(movies_by_year) +
  aes(x = ..., y = ...) +
  geom_...(linewidth = 1) +
  labs(
    title = "...",
    x = "Year",
    y = "Number of movies released",
    caption = "Source: Data 8, movies_by_year"
  ) +
  theme_minimal(base_size = 16)

movies_line

In [ ]:
. <- ottr::check("tests/q6_1.R")

<br/><br/>

<hr style="border: 1px solid #fdb515;" />

## 7. Matching research questions to plots

This is the part that matters for your blog assignments. You will have a question and a dataset,
and you need to know whether the data can say anything about the question **before**
you run a test on it.

Start from the variables. What type is the x-variable, what type is the y-variable,
and what are you actually asking?

| Your question | X variable | Y variable | Plot |
|---|---|---|---|
| What does this variable look like? How spread out is it? | numeric | none, it shows counts | **histogram** |
| Is there an association between these two things? | numeric | numeric | **scatter plot** |
| Do these groups differ? | numeric, split by group | none, it shows counts | **histogram**, faceted or overlaid |
| Which category has more? | categorical | count, or a value | **bar chart** |
| Is this changing over time? | date, time, or year | numeric | **line chart** |

Research questions for your posts might be framed like:

- **"Is there an association between two variables?"** is a scatter plot. If the
  points have no shape, the data cannot answer the question, and it is much better to
  find that out now than after you have built a post around it.
- **"Is there a difference between two groups?"** is a histogram split by group. If
  the two distributions sit almost on top of each other, there is not much of a
  difference to go looking for.
- **"Is this variable's average different from some known value?"** is a histogram of
  that one variable, with the value you are comparing against marked on it. 

---

**Question 7.1.** *Do taller mothers have heavier babies?*

Which plot answers this most appropriately?

- `A.` Histogram of `birth_weight`
- `B.` Scatter plot of `birth_weight` against `maternal_height`
- `C.` Bar chart of `maternal_smoker`
- `D.` Line chart of `birth_weight` over `maternal_age`

Set `q7_1_answer` to the letter, as a string.

In [ ]:
q7_1_answer <- NULL # YOUR CODE HERE

q7_1_answer

In [ ]:
. <- ottr::check("tests/q7_1.R")

---

**Question 7.2.** *Are pregnancies shorter for mothers who smoked than for mothers who
did not?*

Which plot answers this?

- `A.` Scatter plot of `gestational_days` against `birth_weight`
- `B.` Bar chart counting how many mothers are in each smoking group
- `C.` Histogram of `gestational_days`, faceted or filled by `maternal_smoker`
- `D.` Line chart of `gestational_days` over `maternal_age`

Set `q7_2_answer` to the letter, as a string.

In [ ]:
q7_2_answer <- NULL # YOUR CODE HERE

q7_2_answer

In [ ]:
. <- ottr::check("tests/q7_2.R")

---

**Question 7.3.** *Which studio has the most films in the top 200?*

Which plot answers this?

- `A.` Bar chart of `studio`
- `B.` Histogram of `gross`
- `C.` Scatter plot of `gross` against `year`
- `D.` Line chart of `gross` against `year`

Set `q7_3_answer` to the letter, as a string.

In [ ]:
q7_3_answer <- NULL # YOUR CODE HERE

q7_3_answer

In [ ]:
. <- ottr::check("tests/q7_3.R")

---

**Question 7.4.** Your turn, with no scaffolding.

Pick **one** of these two questions and build the plot that answers it. Name it
`my_plot`. It needs a title, both axis labels, and a theme.

1. *Is there an association between the mother's pre-pregnancy weight
   (`maternal_pregnancy_weight`) and the baby's birth weight (`birth_weight`)?*
2. *Do the mothers who smoked have a different age distribution (`maternal_age`)
   than the mothers who did not?*

Read the question first, decide which row of the table above it lands in, and then
write the plot.

In [ ]:
my_plot <- NULL # YOUR CODE HERE

my_plot

In [ ]:
. <- ottr::check("tests/q7_4.R")

---

**Question 7.5.** In the cell below, write two or three sentences: which question you
picked, what the plot shows, and whether this data looks like it could answer the
question.

This one is read by a human, not by the autograder.

*Replace this line with your answer.*

<br/><br/>

<hr style="border: 1px solid #fdb515;" />

## 8. Putting an image in a post

Back to Section 1. You have an image, and you need it to show up in a blog post. This is where the path rules from Section 1 get used.

First, one thing this section is **not** about.

<div style="border-left: 4px solid #fdb515; background: #fffdf5; padding: 0.8em 1em; margin: 1em 0;">
<b>Plots do not work this way.</b>
<p>In your blog, a plot comes from a code chunk and draws itself when the post renders,
with its caption and alt text (the text that shows if the image does not) set as
<a href="https://quarto.org/docs/computations/execution-options.html" target="_blank">chunk</a>
options. Do not save a plot to a file and then embed the file. The moment you change
the data, the code produces a new plot and the saved copy stays old, and now your post
shows a figure that no longer matches the analysis next to it.</p>
<p><code>ggsave()</code> exists for when you need a figure <b>outside</b> the render:
a slide, a paper, something you are emailing to someone. Not for your own post.</p>
</div>

This section is about every **other** image. A photo, a screenshot, a diagram, a
map, something you downloaded. Those are real files sitting in a folder, and the only
way your post can find one is if you give it the right path.

Look at what is in the `images` folder next to this notebook:

In [ ]:
list.files("images")

---

**Question 8.1.** Pick any one of those files. Set `image_path` to the **relative** path
from this notebook to that file, as a string.

For example, if the folder contained `cat.png`, the answer would be
`"images/cat.png"`. Not `/home/jovyan/lab03/images/cat.png`, and not just `"cat.png"`.

In [ ]:
image_path <- NULL # YOUR CODE HERE

image_path

In [ ]:
. <- ottr::check("tests/q8_1.R")

You can check a path before you trust it. `file.exists()` returns `TRUE` if R can
find the file and `FALSE` if it cannot. When an image will not show up, this is the
fastest way to find out whether the problem is the path or something else.

In [ ]:
file.exists(image_path)

---

**Question 8.2.** Now use it.

The markdown cell below has an image reference with the path missing. Replace
`FILL_IN_THE_PATH_HERE` with the path you just worked out, then run the cell. Unfortunately, you can't use the variable name here. The
image should appear underneath. A broken image icon means the path is wrong.

The syntax is `![alt text](path)`. The alt text describes the image for anyone using a
screen reader, and it is not optional in your blog posts. Replace the placeholder
text with a real description of whatever image you picked.

This question is checked by a human, not by the autograder.

**Edit the line below, then run this cell:**

![Replace this with a description of the image](FILL_IN_THE_PATH_HERE)

---

**Question 8.3.** One more. This is the one students most often get wrong.

Back to your blog, with the structure from Section 1:

```
myblog/                 # your project folder
├─ _quarto.yml
├─ index.qmd
├─ about.qmd
├─ posts/               # folder where posts live
│  ├─ _metadata.yml
│  ├─ post1/            # new sub-directory for the post
│  │  └─ index.qmd      # the actual post file
│  └─ ...               # (other posts go here eventually)
└─ is_me.jpeg           # the image for your About page
```

In Question 1.3 you pointed at `is_me.jpeg` from `about.qmd`, which was easy because
they sit in the same folder. Now you are writing `posts/post1/index.qmd`, and you want
to show that same image in the post.

The path is read starting from `post1`. `is_me.jpeg` is not in `post1`, and it is not
in `posts` either. Count how many levels you have to climb. Move out of a directory with `../`

Set `post_image_path` to the path that goes inside `posts/post1/index.qmd`.



In [ ]:
post_image_path <- NULL # YOUR CODE HERE

post_image_path

In [ ]:
. <- ottr::check("tests/q8_3.R")

That is the rule. Work out where the file sits relative to the thing pointing
at it, and write that. Get it right and the image works on your laptop, on the server,
and for every reader. Write an absolute path and it works in exactly one place.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />

You're done with Lab 3!

Here's what you can now do: load the tidyverse, read a CSV with a path that will still
work on somebody else's machine, and build the four plots that answer the four
questions you are most likely to ask. More importantly, you can look at a research
question and know which plot to use, then look at that picture and tell
whether the data has anything to say. 
    
**Important submission information:** Be sure to run the tests and verify that they all
pass, then choose **Save Notebook** from the **File** menu, then Download the .ipynb.
Then, go to your course's Gradescope and submit the corresponding assignment. The name
of this assignment is "Lab 03".

### Athletic support of this data science course

Erling Haaland and Julian Alaphilippe are impressed by your skills!

<img src="erling_haaland.png" alt="Famous footballer Erling Haaland" width="300"/>

<img src="julian_alaphilippe.jpeg" alt="Famous cyclist Julian Alaphilippe.jpeg" width="300"/>

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
# Re-run all the autograder tests in the tests/ folder.
. <- sapply(list.files("tests", pattern = "\\.R$", full.names = TRUE), function(f) ottr::check(f))

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. Click Download in the bar at the top of the notebook. **Please save before downloading!** Submit your final version to Gradescope.